In [2]:
# --------------------------------------------------------
# OctFormer: Octree-based Transformers for 3D Point Clouds
# Copyright (c) 2023 Peng-Shuai Wang <wangps@hotmail.com>
# Licensed under The MIT License [see LICENSE for details]
# Written by Peng-Shuai Wang
# ------------------------------------------------------
import torch.nn.functional as F

import ocnn
import torch
import torch.nn as nn
from ocnn.octree import Octree
from typing import List

from models.octformer import OctFormer



class ClsHeader(nn.Module):
  def __init__(self, out_channels: int, in_channels: int,
                nempty: bool = False, dropout: float = 0.5):
      super().__init__()
      self.global_pool = ocnn.nn.OctreeGlobalPool(nempty)
      # Increase input dimension by 1 to account for the ptau scalar
      self.cls_header = nn.Sequential(
          ocnn.modules.FcBnRelu(in_channels + 1, 128),
          nn.Dropout(p=dropout),
          nn.Linear(128, out_channels)
      )

  def forward(self, data: torch.Tensor, octree: Octree, depth: int, ptau: torch.Tensor):
      # Global pooling: produces a feature vector of shape [batch_size, in_channels]
      pooled = self.global_pool(data, octree, depth)
      # Ensure ptau is shape [batch_size, 1]
      if ptau.dim() == 1:
          ptau = ptau.unsqueeze(1)
      # Concatenate pooled features with ptau along the feature dimension.
      combined = torch.cat([pooled, ptau.float()], dim=1)
      logit = self.cls_header(combined)
      return logit


class OctFormerCls(torch.nn.Module):

  def __init__(self, in_channels: int, out_channels: int,
               channels: List[int] = [3200, 3200, 6400, 64],
               num_blocks: List[int] = [2, 2, 2, 2],
               num_heads: List[int] = [6, 6, 6, 6],
               patch_size: int = 0, dilation: int = 0,
               drop_path: float = 0.5, nempty: bool = True,
               stem_down: int = 2, head_drop: float = 0.5, **kwargs):
    super().__init__()
    self.biomarker_fc = nn.Linear(1, 1) 
    self.fc = nn.Linear(out_channels + 1, out_channels)
    self.backbone = OctFormer(
        in_channels, channels, num_blocks, num_heads, patch_size, dilation,
        drop_path, nempty, stem_down)
    self.head = ClsHeader(
        out_channels, channels[-1], nempty, head_drop)
    self.apply(self.init_weights)

  def init_weights(self, m):
    if isinstance(m, torch.nn.Linear):
      torch.nn.init.trunc_normal_(m.weight, std=0.02)
      if isinstance(m, torch.nn.Linear) and m.bias is not None:
        torch.nn.init.constant_(m.bias, 0)

  def forward(self, data: torch.Tensor, octree: Octree, depth: int, ptau:torch.Tensor):
    
    features = self.backbone(data, octree, depth)
    curr_depth = min(features.keys())
    output = self.head(features[curr_depth], octree, curr_depth, ptau)
    # print(output.shape)
    # if ptau.dim() == 1:
    #     ptau = ptau.unsqueeze(1)
    # Process the 1D ptau using the biomarker fully connected layer.
    # biomarker_features = self.biomarker_fc(ptau.float())  # Shape: [batch_size, 1]

    # # Concatenate the head output with the ptau features.
    # combined_features = torch.cat([output, biomarker_features], dim=1)  # Shape: [batch_size, out_channels + 1]
    # # Pass the combined features through the final fully connected layer.
    # final_logits = self.fc(combined_features)
    # print(final_logits.shape)
    return output


In [ ]:
import ocnn


In [ ]:
%config Completer.use_jedi = True


In [3]:
import os
import math
import pandas as pd
import trimesh
import numpy as np
import cyminiball  # Optional; only used in the original normalization stage.
from tqdm.notebook import tqdm  # Use the notebook version of tqdm for Jupyter.
from typing import Optional
from plyfile import PlyData, PlyElement
import fpsample  # Ensure fpsample is installed and accessible


import os
import numpy as np
from plyfile import PlyData, PlyElement
from typing import Optional
def read_tetgen_files(folder_path: str):
    """
    Reads TetGen files from the given folder.
    
    Expects one .node file and one .ele file.
    
    Returns:
      vertices: np.ndarray of shape (N, 3)
      tets: np.ndarray of shape (M, 4) (tetrahedra connectivity, 0-indexed)
    """
    node_file = None
    ele_file = None
    for f in os.listdir(folder_path):
        if f.endswith('.node'):
            node_file = os.path.join(folder_path, f)
        elif f.endswith('.ele'):
            ele_file = os.path.join(folder_path, f)
    
    if node_file is None or ele_file is None:
        raise ValueError(f"Missing .node or .ele file in folder {folder_path}")
    
    # Read .node file
    with open(node_file, 'r') as fid:
        lines = fid.readlines()
    header = lines[0].strip().split()
    num_nodes = int(header[0])
    vertices = []
    for line in lines[1:num_nodes+1]:
        parts = line.strip().split()
        # Expecting: id x y z [optional attributes]
        vertices.append([float(parts[1]), float(parts[2]), float(parts[3])])
    vertices = np.array(vertices)
    
    # Read .ele file
    with open(ele_file, 'r') as fid:
        lines = fid.readlines()
    header = lines[0].strip().split()
    num_tets = int(header[0])
    tets = []
    for line in lines[1:num_tets+1]:
        parts = line.strip().split()
        # Expecting: id n1 n2 n3 n4 [optional attributes]
        tet = [int(p) for p in parts[1:5]]
        # Convert from 1-indexed to 0-indexed.
        tet = [i - 1 for i in tet]
        tets.append(tet)
    tets = np.array(tets)
    return vertices, tets
vertices, tets = read_tetgen_files("/home/local/ASURITE/mfarazi/Mohammad/ColabNotebooks/ColabNotebooks/data/ones/m002S4654L053019M33TCF")

In [ ]:
import lapy
from lapy import TetMesh
from lapy import Solver

T = TetMesh(vertices, tets+1)

fem = Solver(T, lump=True)

evals, evec = fem.eigs(50)

TetMesh with regular Laplace
Solver: spsolve (LU decomposition) ...


In [ ]:
import numpy as np

# import plotly
# plotly.offline.init_notebook_mode(connected=True)
import plotly.io as pio
from lapy.diffgeo import compute_divergence, compute_gradient

from lapy import TetMesh
from lapy.plot import plot_tet_mesh
A, B = fem.stiffness, fem.mass
Bi = B.copy()
Bi.data **= -1

grad = compute_gradient(T, evec[:, evnum])
divx = -compute_divergence(T, grad)
vfunc = Bi * divx
print(vfunc.shape)

(157703,)


In [ ]:
from lapy import TetMesh, TriaMesh, shapedna
# compute eigenvalues and eigenvectors for tet mesh
evTet = shapedna.compute_shapedna(T, k=128)
evTet["Eigenvectors"]
evTet["Eigenvalues"]

TetMesh with regular Laplace
Solver: spsolve (LU decomposition) ...


array([7.28583860e-17, 1.44940618e-04, 3.52308677e-04, 4.40742110e-04,
       6.48290481e-04, 7.27501895e-04, 7.91828323e-04, 1.18593306e-03,
       1.26240691e-03, 1.38452087e-03, 1.50757957e-03, 1.72000524e-03,
       1.81859576e-03, 1.91645502e-03, 2.16571950e-03, 2.30440053e-03,
       2.36262216e-03, 2.57131147e-03, 2.65518708e-03, 2.82847284e-03,
       3.08888107e-03, 3.24926785e-03, 3.32304524e-03, 3.40864068e-03,
       3.59173199e-03, 3.70353960e-03, 3.80653325e-03, 3.98487656e-03,
       4.21707179e-03, 4.37847982e-03, 4.48790336e-03, 4.71073717e-03,
       4.84268861e-03, 4.92484136e-03, 5.10925267e-03, 5.29878245e-03,
       5.36337290e-03, 5.44805173e-03, 5.68133835e-03, 5.81054656e-03,
       6.02979505e-03, 6.19333439e-03, 6.26783606e-03, 6.41369196e-03,
       6.55104922e-03, 6.63385277e-03, 6.83253696e-03, 7.01679232e-03,
       7.12532633e-03, 7.36488636e-03, 7.48966496e-03, 7.58681973e-03,
       7.72473353e-03, 8.00815256e-03, 8.13862627e-03, 8.36556779e-03,
      

In [ ]:
import numpy as np

def compute_hks(ev, num_t=16, t_min=0.01, t_max=1.0):
    """
    Compute the Heat Kernel Signature (HKS) given ShapeDNA eigenvalues and eigenvectors.

    Parameters:
    ----------
    ev : dict
        Dictionary containing 'Eigenvalues' and 'Eigenvectors'.
    num_t : int, default=100
        Number of time steps for the HKS.
    t_min : float, default=0.01
        Minimum time scale.
    t_max : float, default=1.0
        Maximum time scale.

    Returns:
    -------
    hks : np.ndarray
        HKS matrix of shape (num_vertices, num_t).
    """

    eigenvalues = ev['Eigenvalues']  # (k,)
    eigenvectors = ev['Eigenvectors']  # (num_vertices, k)

    num_vertices, k = eigenvectors.shape

    # Generate logarithmically spaced time values
    time_scales = np.logspace(np.log10(t_min), np.log10(t_max), num_t)

    # Compute HKS
    hks = np.zeros((num_vertices, num_t))

    for i, t in enumerate(time_scales):
        exp_term = np.exp(-eigenvalues * t)  # (k,)
        hks[:, i] = np.sum((eigenvectors ** 2) * exp_term[None, :], axis=1)

    return hks, time_scales
hks, time_scales = compute_hks(evTet, num_t=16, t_min=0.001, t_max=3)



In [ ]:
# hks /= np.max(hks, axis=1, keepdims=True)
hks[90000:90001]*10000


array([[5.26644582, 5.26641213, 5.26635469, 5.26625674, 5.2660897 ,
        5.26580486, 5.26531917, 5.26449104, 5.26307924, 5.26067288,
        5.2565728 , 5.24959117, 5.2377152 , 5.21754968, 5.18341191,
        5.12591752]])

In [ ]:
shapedna.normalize_ev(T, evTet["Eigenvalues"], method="geometry")
shapedna.reweight_ev(evTet["Eigenvalues"])


Found 237752 triangles on boundary.


array([7.28583860e-17, 7.24703092e-05, 1.17436226e-04, 1.10185527e-04,
       1.29658096e-04, 1.21250316e-04, 1.13118332e-04, 1.48241633e-04,
       1.40267435e-04, 1.38452087e-04, 1.37052688e-04, 1.43333770e-04,
       1.39891981e-04, 1.36889644e-04, 1.44381300e-04, 1.44025033e-04,
       1.38977774e-04, 1.42850637e-04, 1.39746689e-04, 1.41423642e-04,
       1.47089575e-04, 1.47693993e-04, 1.44480228e-04, 1.42026695e-04,
       1.43669280e-04, 1.42443831e-04, 1.40982713e-04, 1.42317020e-04,
       1.45416268e-04, 1.45949327e-04, 1.44771076e-04, 1.47210537e-04,
       1.46748140e-04, 1.44848275e-04, 1.45978648e-04, 1.47188401e-04,
       1.44956024e-04, 1.43369782e-04, 1.45675342e-04, 1.45263664e-04,
       1.47068172e-04, 1.47460343e-04, 1.45763629e-04, 1.45765726e-04,
       1.45578871e-04, 1.44214191e-04, 1.45373127e-04, 1.46183173e-04,
       1.45414823e-04, 1.47297727e-04, 1.46856176e-04, 1.45900379e-04,
       1.45749689e-04, 1.48299121e-04, 1.47975023e-04, 1.49385139e-04,
      

In [ ]:
shapedna.reweight_ev(evTet["Eigenvalues"])

array([6.24500451e-17, 7.24703092e-05, 1.17436226e-04])

In [ ]:
# --------------------------------------------------------
# Octree-based Sparse Convolutional Neural Networks
# Adapted for Jupyter Notebook
# Copyright (c) 2022 Peng-Shuai Wang <wangps@hotmail.com>
# Licensed under The MIT License [see LICENSE for details]
# Written by Peng-Shuai Wang (Adapted by You)
# --------------------------------------------------------

import os
import sys
import shutil
import argparse

from datetime import datetime
from yacs.config import CfgNode as CN

# Initialize the configuration node
_C = CN(new_allowed=True)

_C.BASE = ['']

# SOLVER related parameters
_C.SOLVER = CN(new_allowed=True)
_C.SOLVER.alias             = ''         # The experiment alias
_C.SOLVER.gpu               = (0,)       # The gpu ids
_C.SOLVER.run               = 'train'    # Choose from train or test

_C.SOLVER.logdir            = 'logs'     # Directory where to write event logs
_C.SOLVER.ckpt              = ''         # Restore weights from checkpoint file
_C.SOLVER.ckpt_num          = 10         # The number of checkpoints kept

_C.SOLVER.type              = 'sgd'      # Choose from sgd or adam
_C.SOLVER.weight_decay      = 0.0005     # The weight decay on model weights
_C.SOLVER.clip_grad         = -1.0       # Clip gradient norm (-1: disable)
_C.SOLVER.max_epoch         = 300        # Maximum training epoch
_C.SOLVER.warmup_epoch      = 20         # The warmup epoch number
_C.SOLVER.warmup_init       = 0.001      # The initial ratio of the warmup
_C.SOLVER.eval_epoch        = 1          # Maximum evaluating epoch
_C.SOLVER.eval_step         = -1         # Maximum evaluating steps
_C.SOLVER.test_every_epoch  = 10         # Test model every n training epochs
_C.SOLVER.log_per_iter      = -1         # Output log every k training iterations
_C.SOLVER.best_val          = 'min:loss' # The best validation metric

_C.SOLVER.lr_type           = 'step'     # Learning rate type: step or cos
_C.SOLVER.lr                = 0.1        # Initial learning rate
_C.SOLVER.lr_min            = 0.0001     # The minimum learning rate
_C.SOLVER.gamma             = 0.1        # Learning rate step-wise decay
_C.SOLVER.milestones        = (120, 180,)# Learning rate milestones
_C.SOLVER.lr_power          = 0.9        # Used in poly learning rate

_C.SOLVER.dist_url          = 'tcp://localhost:10001'
_C.SOLVER.progress_bar      = True       # Enable the progress bar or not
_C.SOLVER.rand_seed         = -1         # Fix the random seed if larger than 0
_C.SOLVER.empty_cache       = True       # Empty cuda cache periodically

# DATA related parameters
_C.DATA = CN(new_allowed=True)
_C.DATA.train = CN(new_allowed=True)
_C.DATA.train.name          = ''          # The name of the dataset
_C.DATA.train.disable       = False       # Disable this dataset or not
_C.DATA.train.pin_memory    = True

# For octree building
_C.DATA.train.depth         = 5           # The octree depth
_C.DATA.train.full_depth    = 2           # The full depth
_C.DATA.train.adaptive      = False       # Build the adaptive octree

# For transformation
_C.DATA.train.orient_normal = ''          # Used to re-orient normal directions

# For data augmentation
_C.DATA.train.distort       = False       # Whether to apply data augmentation
_C.DATA.train.scale         = 0.0         # Scale the points
_C.DATA.train.uniform       = False       # Generate uniform scales
_C.DATA.train.jitter        = 0.0         # Jitter the points
_C.DATA.train.interval      = (1, 1, 1)   # Use interval & angle to generate random angle
_C.DATA.train.angle         = (180, 180, 180)
_C.DATA.train.flip          = (0.0, 0.0, 0.0)

# For data loading
_C.DATA.train.location      = ''          # The data location
_C.DATA.train.filelist      = ''          # The data filelist
_C.DATA.train.batch_size    = 32          # Training data batch size
_C.DATA.train.take          = -1          # Number of samples used for training
_C.DATA.train.num_workers   = 4           # Number of workers to load the data
_C.DATA.train.shuffle       = False       # Shuffle the input data
_C.DATA.train.in_memory     = False       # Load the training data into memory

_C.DATA.test = _C.DATA.train.clone()
_C.DATA.test.num_workers    = 2

# MODEL related parameters
_C.MODEL = CN(new_allowed=True)
_C.MODEL.name               = ''          # The name of the model
_C.MODEL.feature            = 'ND'        # The input features
_C.MODEL.channel            = 3           # The input feature channel
_C.MODEL.nempty             = False       # Perform Octree Conv on non-empty octree nodes

_C.MODEL.sync_bn            = False       # Use sync_bn when training the network
_C.MODEL.use_checkpoint     = False       # Use checkpoint to save memory
_C.MODEL.find_unused_parameters = False   # Used in DistributedDataParallel

# LOSS related parameters
_C.LOSS = CN(new_allowed=True)
_C.LOSS.name                = ''          # The name of the loss
_C.LOSS.num_class           = 40          # The class number for the cross-entropy loss
_C.LOSS.label_smoothing     = 0.0         # The factor of label smoothing

# Backup commands
_C.SYS = CN(new_allowed=True)
_C.SYS.cmds              = ''             # Used to backup the commands

FLAGS = _C

def _load_from_file(filename):
    """
    Load configuration files recursively.
    """
    cfgs = []
    bases = [filename]
    while len(bases) > 0:
        base = bases.pop(0)
        if base:
            with open(base, 'r') as fid:
                cfg = CN.load_cfg(fid)
            cfgs.append(cfg)
            if 'BASE' in cfg:
                bases += cfg.BASE
    cfgs.reverse()
    return cfgs

def update_config(config, config_file=None, opts=None):
    """
    Update the configuration using a config file and options.
    
    Args:
        config (CfgNode): The configuration node to update.
        config_file (str): Path to the configuration file.
        opts (list): List of options to override, e.g., ['SOLVER.lr', '0.01'].
    
    Returns:
        Updated and frozen configuration.
    """
    config.defrost()
    if config_file:
        cfgs = _load_from_file(config_file)
        for cfg in cfgs:
            config.merge_from_other_cfg(cfg)
    if opts:
        config.merge_from_list(opts)
    config.SYS.cmds = ' '.join(sys.argv)
    
    # Update logdir based on alias and current time if needed
    alias = config.SOLVER.alias.lower()
    if 'time' in alias:  # 'time' is a special keyword
        alias = alias.replace('time', datetime.now().strftime('%m%d%H%M'))
    if alias != '':
        config.SOLVER.logdir += '_' + alias
    config.freeze()
    return config

def backup_config(config, config_file=None):
    """
    Back up the configuration and the config file into the log directory.
    
    Args:
        config (CfgNode): The configuration node.
        config_file (str): Path to the configuration file.
    """
    logdir = config.SOLVER.logdir
    os.makedirs(logdir, exist_ok=True)
    
    # Copy the configuration file to the log directory if provided
    if config_file:
        shutil.copy2(config_file, logdir)
    
    # Dump all configs to a file
    filename = os.path.join(logdir, 'all_configs.yaml')
    with open(filename, 'w') as fid:
        fid.write(config.dump())

def get_config():
    """
    Get the default configuration.
    """
    return FLAGS

def _set_env_var(config):
    """
    Set the CUDA_VISIBLE_DEVICES environment variable based on the config.
    """
    gpus = ','.join([str(a) for a in config.SOLVER.gpu])
    os.environ['CUDA_VISIBLE_DEVICES'] = gpus

def get_config():
    """
    Return the default configuration.
    """
    return FLAGS

def parse_args(backup=True):
    parser = argparse.ArgumentParser(description='The configs')
    parser.add_argument('--config', type=str,
                        help='experiment configure file name')
    parser.add_argument('opts', nargs=argparse.REMAINDER,
                        help="Modify config options using the command-line")
    
    # Use parse_known_args to ignore extra arguments from the Jupyter kernel.
    args, unknown = parser.parse_known_args()
    
    update_config(FLAGS, config_file=args.config, opts=args.opts)
    
    if backup:
        backup_config(FLAGS, config_file=args.config)
    # Optionally, set environment variables:
    # _set_env_var(FLAGS)
    return FLAGS


# This block is only executed when running as a script.
# In a Jupyter Notebook, you can simply call the functions directly.
# if __name__ == '__main__':
#     flags = parse_args(backup=False)
#     print(flags)

cfg = get_config()
cfg = update_config(cfg, config_file="/home/local/ASURITE/mfarazi/Mohammad/ColabNotebooks/ColabNotebooks/octformer/configs/cls_tet.yaml", 
                    opts=['DATA.train.location', 'data/tet'])
print(cfg.MODEL)



channel: 3
feature: P
find_unused_parameters: False
name: octformercls
nempty: False
nout: 2
sync_bn: False
use_checkpoint: False


In [ ]:
def get_input_feature( octree, flag):
    flags = flag.MODEL
    # print(flags)
    octree_feature = ocnn.modules.InputFeature(flags.feature, flags.nempty)
    data = octree_feature(octree)
    return data

In [ ]:
import torch
from torch.utils.data import Sampler, DistributedSampler, Dataset


class InfSampler(Sampler):
  def __init__(self, dataset: Dataset, shuffle: bool = True) -> None:
    self.dataset = dataset
    self.shuffle = shuffle
    self.reset_sampler()

  def reset_sampler(self):
    num = len(self.dataset)
    indices = torch.randperm(num) if self.shuffle else torch.arange(num)
    self.indices = indices.tolist()
    self.iter_num = 0

  def __iter__(self):
    return self

  def __next__(self):
    value = self.indices[self.iter_num]
    self.iter_num = self.iter_num + 1

    if self.iter_num >= len(self.indices):
      self.reset_sampler()
    return value

  def __len__(self):
    return len(self.dataset)

In [ ]:
import torch
import numpy as np
# from .sampler import InfSampler, DistributedInfSampler
from thsolver import Dataset
from ocnn.octree import Points
from ocnn.dataset import CollateBatch
import pdb
from datasets.utils import ReadPly, Transform

from datasets.tetmesh import TetTransform, read_file, get_tetmesh_dataset
flag = cfg.DATA.test

# Define the device (use 'cuda' if available, otherwise fallback to CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize transforms, collate function, and dataset
transform = TetTransform(flag)
collate_batch = CollateBatch()

dataset = Dataset(flag.location, flag.filelist, transform,
                  read_file=read_file, take=flag.take)

# Initialize the model and move it to the target device
model = OctFormerCls(
    3, 3,
    channels=[32, 32],
    num_blocks=[2, 2],
    num_heads=[2, 2],
    patch_size=32, dilation=1,
    drop_path=0.0, nempty=False,
    stem_down=2, head_drop=0.0)
model.to(device)

# Set up your sampler and DataLoader
sampler = InfSampler(dataset, shuffle=True)
data_loader = torch.utils.data.DataLoader(
    dataset, batch_size=1,
    sampler=sampler, collate_fn=collate_batch, pin_memory=False)

# Get the first batch from the DataLoader
batch = next(iter(data_loader))

# Send each element of the batch to the same device
octree = batch['octree'].to(device)
label  = batch['label'].to(device)
ptau   = batch['ptau'].to(device)
# print(data.shape)

# Process input features; if get_input_feature returns a tensor, send it to device too.
data = get_input_feature(octree, cfg)
print(data.shape)

# Now run the model forward pass on the device
# logits = model(data.to(device), octree, octree.depth, ptau)


torch.Size([11336, 3])


In [ ]:
points_obj = octree.to_points(rescale=True)

# `points_obj` is an instance of the Points class. You can then access:
print("XYZ Coordinates:", points_obj.points.shape)
# print("Normals:", points_obj.normals)
# print("Features:", points_obj.features)
non_zero_rows = data[~(data == 0).all(dim=1)]
print(non_zero_rows)

XYZ Coordinates: torch.Size([8041, 3])
tensor([[-0.0707, -0.5047, -0.1946],
        [-0.0974, -0.5007, -0.1354],
        [-0.0693, -0.5169, -0.1682],
        ...,
        [ 0.2515,  0.5200,  0.1197],
        [ 0.2529,  0.5128,  0.1366],
        [ 0.2510,  0.5111,  0.1581]], device='cuda:0')


In [5]:
def generate_tetrahedron_strips(tetrahedra):
    """
    Generate tetrahedron strips from a list of tetrahedra.
    
    Each tetrahedron is a tuple (v0, v1, v2, v3) representing vertex indices.
    A tetra strip is a sequence of vertices [v0, v1, v2, v3, v4, ...]
    where the first tetrahedron is (v0, v1, v2, v3) and each subsequent tetrahedron
    is (v1, v2, v3, v_new), sharing the last 3 vertices.
    
    Parameters:
        tetrahedra (list of tuples): List of tetrahedra.
        
    Returns:
        list of lists: Each inner list is a tetrahedron strip (a sequence of vertex indices).
    """
    # Build a dictionary mapping a face (3 vertices, order independent) to a list of tetrahedron indices.
    face_to_tets = {}
    for idx, tet in enumerate(tetrahedra):
        faces = [
            (tet[0], tet[1], tet[2]),
            (tet[0], tet[1], tet[3]),
            (tet[0], tet[2], tet[3]),
            (tet[1], tet[2], tet[3])
        ]
        for face in faces:
            key = frozenset(face)  # order-independent key
            face_to_tets.setdefault(key, []).append(idx)
    
    used = [False] * len(tetrahedra)
    strips = []
    
    # Greedily build tetrahedron strips.
    for i, tet in enumerate(tetrahedra):
        if used[i]:
            continue
        
        # Start a new strip with the tetrahedron's vertices.
        strip = list(tet)
        used[i] = True
        
        # Try extending the strip by looking for a tetrahedron that shares the last three vertices.
        while True:
            last_three = tuple(strip[-3:])
            key = frozenset(last_three)
            found_extension = False
            
            for tet_idx in face_to_tets.get(key, []):
                if used[tet_idx]:
                    continue
                candidate_tet = tetrahedra[tet_idx]
                # Check if candidate tetrahedron contains all vertices in last_three.
                if all(v in candidate_tet for v in last_three):
                    # Find the vertex in candidate_tet that is not in last_three.
                    for v in candidate_tet:
                        if v not in last_three:
                            new_vertex = v
                            break
                    strip.append(new_vertex)
                    used[tet_idx] = True
                    found_extension = True
                    break  # Extend one tetrahedron at a time.
            if not found_extension:
                break
        
        strips.append(strip)
    
    return strips


# Example usage:
if __name__ == "__main__":
    # Define a simple tetrahedral mesh.
    # For instance, consider two tetrahedra sharing a common face:
    # Tetrahedron 1: (0, 1, 2, 3)
    # Tetrahedron 2: (1, 2, 3, 4)  # Shares face (1, 2, 3) with the first one.
    tetrahedra = [
        (0, 1, 2, 3),
        (1, 2, 3, 4)
    ]
    
    strips = generate_tetrahedron_strips(tets)
    print("Generated Tetrahedron Strips:")
    for s in strips:
        print(s)


In [4]:
vertices, tets = read_tetgen_files("/home/local/ASURITE/mfarazi/Mohammad/ColabNotebooks/ColabNotebooks/data/ones/m002S4654L053019M33TCF")